# 🤖 Agent Research Platform — Google Colab

Sigue las celdas **en orden**. Al final tendrás un enlace público para abrir la plataforma en el navegador.

**Requisitos previos:**
- Una cuenta gratuita en [ngrok.com](https://ngrok.com) (para obtener el authtoken)
- Tu `GEMINI_API_KEY` de [aistudio.google.com](https://aistudio.google.com/apikey)


## Paso 1 — Clonar el repositorio

In [ ]:
import os, subprocess

REPO_DIR = '/content/RepositorioJota'
BRANCH   = 'claude/agent-content-platform-SOyKj'

if os.path.exists(REPO_DIR):
    print('📂 Carpeta existente — actualizando...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
    print('✅ Repositorio actualizado al último commit')
else:
    print('📥 Clonando repositorio...')
    subprocess.run(['git', 'clone', 'https://github.com/jotadoliveira22/RepositorioJota.git', REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    print('✅ Repositorio clonado')

os.chdir(REPO_DIR)
print(f'📌 Commit activo: {subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-1"]).decode().strip()}')

## Paso 2 — Instalar dependencias

In [ ]:
!pip install -q fastapi uvicorn[standard] httpx python-dotenv pydantic aiofiles python-multipart pyngrok
print('✅ Dependencias instaladas')

## Paso 3 — Configurar las claves API

**Opción A (recomendada) — Colab Secrets:**
1. En el panel izquierdo de Colab haz clic en el ícono 🔑 **Secrets**
2. Agrega dos secrets:
   - `GEMINI_API_KEY` → tu clave de Google AI Studio
   - `NGROK_TOKEN` → tu authtoken de [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
3. Activa el toggle **"Notebook access"** para cada uno

**Opción B — Escribir directamente en la celda** (menos seguro):
Reemplaza `None` con tus claves entre comillas.

In [ ]:
import os

# --- Opción A: desde Colab Secrets (recomendado) ---
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    print('✅ Claves cargadas desde Colab Secrets')
except Exception:
    # --- Opción B: escribir directamente ---
    os.environ['GEMINI_API_KEY'] = None   # <-- reemplaza None por tu clave entre comillas
    NGROK_TOKEN = None                    # <-- reemplaza None por tu token ngrok entre comillas
    print('⚠️  Usando claves escritas directamente')

# Verificar que están seteadas
assert os.environ.get('GEMINI_API_KEY'), '❌ GEMINI_API_KEY no configurada'
assert NGROK_TOKEN, '❌ NGROK_TOKEN no configurado'
print('✅ Claves verificadas')

## Paso 3b — Verificar API key y modelos disponibles

In [ ]:
import httpx, os

api_key = os.environ.get('GEMINI_API_KEY', '')
print(f'🔑 API Key cargada: {api_key[:8]}...{api_key[-4:]}' if len(api_key) > 12 else f'❌ API Key: "{api_key}"')

MODELS = [
    "gemini-2.0-flash",
    "gemini-2.0-flash-001",
    "gemini-1.5-flash",
    "gemini-1.5-flash-latest",
    "gemini-1.5-pro",
]

print('\n🔍 Probando modelos disponibles...')
working = None
for model in MODELS:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    try:
        r = httpx.post(url, json={"contents": [{"parts": [{"text": "hi"}]}], "generationConfig": {"maxOutputTokens": 1}}, timeout=10)
        status = r.status_code
        if status == 200:
            print(f'  ✅ {model} → OK')
            if not working:
                working = model
        else:
            body = r.text[:120].replace('\n', ' ')
            print(f'  ❌ {model} → HTTP {status} | {body}')
    except Exception as e:
        print(f'  ⚠️  {model} → error: {e}')

print()
if working:
    print(f'✅ Modelo a usar: {working}')
else:
    print('❌ Ningún modelo disponible. Revisa tu API key en https://aistudio.google.com/apikey')

## Paso 4 — Iniciar el servidor y obtener la URL pública

In [ ]:
import sys
import os
import threading
import time
import uvicorn
from pyngrok import ngrok, conf

# Asegurar que Python pueda encontrar el módulo 'backend'
repo_dir = os.path.abspath('/content/RepositorioJota')
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Importar la app directamente (evita problemas de resolución de módulos en hilos)
from backend.main import app as fastapi_app

# Configurar ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Cerrar tunnels previos si los hay
ngrok.kill()

# Crear túnel público
tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print('\n' + '='*55)
print(f'  🌐 ABRE ESTE ENLACE EN TU NAVEGADOR:')
print(f'  👉  {public_url}')
print('='*55 + '\n')

# Iniciar servidor FastAPI en un hilo separado
def run_server():
    uvicorn.run(
        fastapi_app,
        host='0.0.0.0',
        port=8000,
        reload=False,          # reload=False es necesario en Colab
        log_level='warning',   # reduce el ruido en los logs
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # esperar a que arranque

print('✅ Servidor corriendo. La celda debe permanecer activa mientras uses la plataforma.')
print('   Para detener, interrumpe esta celda (botón ■ o Ctrl+M I).')

# Mantener la celda viva
try:
    while True:
        time.sleep(60)
        print(f'   🔄 Activo — {public_url}')
except KeyboardInterrupt:
    ngrok.kill()
    print('\n🛑 Servidor detenido.')

---
## ℹ️ Notas importantes

| Tema | Detalle |
|---|---|
| **Tiempo de vida** | La URL deja de funcionar si cierras la pestaña de Colab o la sesión expira (~12h) |
| **Datos** | Los runs se guardan en `/content/RepositorioJota/data/runs/` dentro de Colab (se pierden al cerrar la sesión) |
| **ngrok gratis** | Permite 1 túnel activo simultáneo. Si ves error de túnel, ejecuta `ngrok.kill()` y vuelve a correr el Paso 4 |
| **Rate limit Gemini** | El pipeline espera automáticamente si llega al límite de la free tier (~40 seg) |
| **API Key** | Nunca escribas tu clave directamente en el notebook si vas a compartirlo. Usa siempre Colab Secrets |
